In [ ]:
!unzip "/content/drive/MyDrive/atm abnormal activity dataset/Liver cyst seg unet data.zip"

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from glob import glob
from typing import Dict, List, Tuple

# ===== CONFIG =====
CLASSES = ["Abscess", "Cyst", "Tumor"]
NUM_CLASSES = len(CLASSES) + 1  # +1 for background
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256
EPOCHS = 30
BATCH_SIZE = 4
LR = 1e-4
SAVE_PRED_SAMPLES = True  # set False if you don’t want saved masks
PRED_SAMPLE_EVERY = 5     # save samples every N epochs

os.makedirs("pred_samples", exist_ok=True)

# ===== Dataset =====
class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_paths = sorted(glob(os.path.join(img_dir, "*.png")) +
                                glob(os.path.join(img_dir, "*.jpg")) +
                                glob(os.path.join(img_dir, "*.jpeg")))
        self.mask_paths = sorted(glob(os.path.join(mask_dir, "*.png")))
        assert len(self.img_paths) == len(self.mask_paths), \
            f"Found {len(self.img_paths)} images but {len(self.mask_paths)} masks. Ensure matching filenames and counts."

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        mask_path = self.mask_paths[idx]

        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        img = img.astype(np.float32) / 255.0
        img_t = torch.from_numpy(img).permute(2, 0, 1)  # (C,H,W)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        # Convert grayscale values (0,85,170,255) → class index (0,1,2,3)
        mapping = {0:0, 85:1, 170:2, 255:3}
        mask_idx = np.vectorize(mapping.get)(mask).astype(np.int64)
        mask_t = torch.from_numpy(mask_idx)  # (H,W), long

        return img_t, mask_t, os.path.basename(img_path)

# ===== U-Net =====
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=4):
        super().__init__()
        self.encoder = nn.ModuleList([
            self.conv_block(in_channels, 64),
            self.conv_block(64, 128),
            self.conv_block(128, 256),
            self.conv_block(256, 512),
        ])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = self.conv_block(512, 1024)
        self.upconvs = nn.ModuleList([
            nn.ConvTranspose2d(1024, 512, 2, 2),
            nn.ConvTranspose2d(512, 256, 2, 2),
            nn.ConvTranspose2d(256, 128, 2, 2),
            nn.ConvTranspose2d(128, 64, 2, 2),
        ])
        self.decoder = nn.ModuleList([
            self.conv_block(1024, 512),
            self.conv_block(512, 256),
            self.conv_block(256, 128),
            self.conv_block(128, 64),
        ])
        self.final = nn.Conv2d(64, out_channels, 1)

    def conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        skips = []
        for enc in self.encoder:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for i in range(len(self.upconvs)):
            x = self.upconvs[i](x)
            skip = skips[i]
            if x.shape[-2:] != skip.shape[-2:]:
                x = nn.functional.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([skip, x], dim=1)
            x = self.decoder[i](x)

        return self.final(x)  # logits (B, C, H, W)

# ===== Metrics =====
@torch.no_grad()
def fast_confusion_matrix(pred: np.ndarray, target: np.ndarray, num_classes: int) -> np.ndarray:
    """
    pred, target: (H,W) integer class ids in [0..num_classes-1]
    returns (num_classes, num_classes) where rows=target, cols=pred
    """
    mask = (target >= 0) & (target < num_classes)
    hist = np.bincount(
        num_classes * target[mask].astype(int) + pred[mask].astype(int),
        minlength=num_classes**2
    ).reshape(num_classes, num_classes)
    return hist

def iou_dice_from_confmat(confmat: np.ndarray) -> Tuple[np.ndarray, np.ndarray, float, float, float]:
    """
    Returns:
      per_class_iou, per_class_dice, mean_iou, mean_dice, pixel_acc
    """
    tp = np.diag(confmat).astype(np.float64)
    fp = confmat.sum(axis=0) - tp
    fn = confmat.sum(axis=1) - tp
    denom_iou = tp + fp + fn
    per_class_iou = np.divide(tp, denom_iou, out=np.zeros_like(tp), where=denom_iou > 0)

    denom_dice = 2*tp + fp + fn
    per_class_dice = np.divide(2*tp, denom_dice, out=np.zeros_like(tp), where=denom_dice > 0)

    mean_iou = np.nanmean(per_class_iou)
    mean_dice = np.nanmean(per_class_dice)
    pixel_acc = tp.sum() / confmat.sum() if confmat.sum() > 0 else 0.0
    return per_class_iou, per_class_dice, float(mean_iou), float(mean_dice), float(pixel_acc)

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion, epoch: int = 0, save_samples: bool = False, sample_dir: str = "pred_samples"):
    model.eval()
    total_loss = 0.0
    confmat = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    # save at most a few samples
    saved = 0
    save_quota = 3

    for imgs, masks, names in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        logits = model(imgs)
        loss = criterion(logits, masks)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)  # (B,H,W)

        # accumulate confmat
        for b in range(preds.shape[0]):
            pred_np = preds[b].detach().cpu().numpy()
            mask_np = masks[b].detach().cpu().numpy()
            confmat += fast_confusion_matrix(pred_np, mask_np, NUM_CLASSES)

        # optionally save some visual predictions
        if save_samples and saved < save_quota:
            for b in range(min(preds.shape[0], save_quota - saved)):
                pmask = preds[b].detach().cpu().numpy().astype(np.uint8)
                # simple color map for 4 classes
                # 0: background(black), 1: red, 2: green, 3: blue
                colors = np.array([
                    [0, 0, 0],
                    [255, 0, 0],
                    [0, 255, 0],
                    [0, 0, 255]
                ], dtype=np.uint8)
                color_mask = colors[pmask]
                out = cv2.cvtColor(color_mask, cv2.COLOR_RGB2BGR)
                out_path = os.path.join(sample_dir, f"epoch{epoch:03d}__{names[b]}")
                cv2.imwrite(out_path.replace(".jpg", "_pred.png").replace(".jpeg", "_pred.png").replace(".png", "_pred.png"), out)
                saved += 1
                if saved >= save_quota:
                    break

    avg_loss = total_loss / max(len(loader), 1)
    per_iou, per_dice, miou, mdice, acc = iou_dice_from_confmat(confmat)
    return {
        "val_loss": avg_loss,
        "per_class_iou": per_iou,
        "per_class_dice": per_dice,
        "miou": miou,
        "mdice": mdice,
        "pixel_acc": acc,
        "confmat": confmat
    }

# ===== Training =====
def plot_curve(xs: List[int], ys: List[float], title: str, ylabel: str, savepath: str):
    plt.figure()
    plt.plot(xs, ys, marker="o")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(savepath, dpi=150)
    plt.close()

def plot_bars(labels: List[str], values: List[float], title: str, ylabel: str, savepath: str, rotate=45):
    plt.figure()
    plt.bar(labels, values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotate, ha="right")
    plt.ylim(0, 1.0)
    plt.grid(True, axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(savepath, dpi=150)
    plt.close()

def train_model(train_loader, val_loader):
    model = UNet(out_channels=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "miou": [],
        "mdice": [],
        "pixel_acc": [],
    }
    # also keep last per-class metrics for bar plots
    last_per_iou = None
    last_per_dice = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for imgs, masks, _ in train_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            loss = criterion(preds, masks)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / max(len(train_loader), 1)

        # validation + metrics
        val_stats = evaluate(
            model, val_loader, criterion,
            epoch=epoch,
            save_samples=SAVE_PRED_SAMPLES and (epoch % PRED_SAMPLE_EVERY == 0)
        )

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_stats["val_loss"])
        history["miou"].append(val_stats["miou"])
        history["mdice"].append(val_stats["mdice"])
        history["pixel_acc"].append(val_stats["pixel_acc"])

        last_per_iou = val_stats["per_class_iou"]
        last_per_dice = val_stats["per_class_dice"]

        print(
            f"Epoch [{epoch}/{EPOCHS}] "
            f"train_loss={train_loss:.4f} | val_loss={val_stats['val_loss']:.4f} | "
            f"mIoU={val_stats['miou']:.4f} | mDice={val_stats['mdice']:.4f} | "
            f"pixAcc={val_stats['pixel_acc']:.4f}"
        )

    # save model
    torch.save(model.state_dict(), "unet_seg.pth")

    # save CSV log
    import csv
    with open("training_log.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "mIoU", "mDice", "pixel_acc"])
        for i in range(len(history["epoch"])):
            writer.writerow([
                history["epoch"][i],
                history["train_loss"][i],
                history["val_loss"][i],
                history["miou"][i],
                history["mdice"][i],
                history["pixel_acc"][i],
            ])

    # plots
    xs = history["epoch"]
    plot_curve(xs, history["train_loss"], "Training Loss", "Loss", "curve_loss.png")
    plot_curve(xs, history["val_loss"], "Validation Loss", "Loss", "curve_val_loss.png")
    plot_curve(xs, history["miou"], "Mean IoU (Validation)", "mIoU", "curve_miou.png")
    plot_curve(xs, history["mdice"], "Mean Dice (Validation)", "mDice", "curve_mdice.png")
    plot_curve(xs, history["pixel_acc"], "Pixel Accuracy (Validation)", "Pixel Acc", "curve_accuracy.png")

    # per-class bar charts for the last epoch
    if last_per_iou is not None:
        labels = ["Background"] + CLASSES
        plot_bars(labels, list(last_per_iou), f"Per-class IoU (Epoch {xs[-1]})", "IoU", f"per_class_iou_epoch_{xs[-1]}.png")
    if last_per_dice is not None:
        labels = ["Background"] + CLASSES
        plot_bars(labels, list(last_per_dice), f"Per-class Dice (Epoch {xs[-1]})", "Dice", f"per_class_dice_epoch_{xs[-1]}.png")

    return model

# ===== Prediction helper =====
@torch.no_grad()
def predict(model: nn.Module, img_path: str, save_path: str = None) -> np.ndarray:
    model.eval()
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    x = torch.from_numpy(img_resized.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    logits = model(x)
    pred = torch.argmax(logits, dim=1)[0].detach().cpu().numpy().astype(np.uint8)

    if save_path is not None:
        colors = np.array([[0,0,0],[255,0,0],[0,255,0],[0,0,255]], dtype=np.uint8)
        color_mask = colors[pred]
        out = cv2.cvtColor(color_mask, cv2.COLOR_RGB2BGR)
        cv2.imwrite(save_path, out)
    return pred

# ===== Run Training =====
if __name__ == "__main__":
    train_dataset = SegDataset(
        r"/content/Liver cyst seg unet data/train/images",
        r"/content/Liver cyst seg unet data/train/masks"
    )
    val_dataset = SegDataset(
        r"/content/Liver cyst seg unet data/valid/images",
        r"/content/Liver cyst seg unet data/valid/masks"
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = train_model(train_loader, val_loader)

    # Example prediction
    pred_mask = predict(
        model,
        r"/content/Liver cyst seg unet data/test/images/DataSet11_Abscess_Arterial_png.rf.b254b6fc8538261db5ba216239d80cec.jpg",
        save_path="example_pred.png"
    )
    print("Saved example prediction to example_pred.png")


In [ ]:
from google.colab.patches import cv2_imshow
import numpy as np
import cv2
import torch

def predict(model, img_path):
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
    tensor = torch.tensor(img_resized, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(tensor)
        pred_class = torch.argmax(pred, dim=1).squeeze().cpu().numpy()

    # Define colors for each class
    colors = {0: (0,0,0), 1: (255,0,0), 2: (0,255,0), 3: (0,0,255)}
    color_mask = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    for cls_idx, color in colors.items():
        color_mask[pred_class == cls_idx] = color

    # Overlay mask on original image
    img_uint8 = (img_resized * 255).astype(np.uint8)
    overlay = cv2.addWeighted(img_uint8, 0.7, color_mask, 0.3, 0)

    # Find predicted classes
    present_classes = [CLASSES[c-1] for c in np.unique(pred_class) if c != 0]
    print("Predicted Classes:", present_classes)

    # Show all
    cv2_imshow(img_uint8)       # Original image
    cv2_imshow(color_mask)      # Mask
    cv2_imshow(overlay)         # Mask overlay
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# Example usage
predict(model, r"/content/Liver cyst seg unet data/test/images/DataSet11_Abscess_Arterial_png.rf.b254b6fc8538261db5ba216239d80cec.jpg")
